In [ ]:
!pip install -q transformers datasets evaluate groq accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.2 MB/s eta 0:00:00


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from groq import Groq
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve
from google.colab import drive
from sklearn.model_selection import train_test_split

In [ ]:
# ==========================================
# KONFIGURACJA I MONTOWANIE DYSKU
# ==========================================
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/SNIPS_OOD_Project"
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

GROQ_API_KEY = ""  # <--- WPISZ TUTAJ SWÓJ KLUCZ
MODEL_NAME = "bert-base-uncased"
NUM_FOLDS = 5
NUM_OOD_SAMPLES = 1750

Mounted at /content/drive


In [ ]:
def generate_unknown_data(n):
    print(f"--- Generowanie {n} przykładów klasy Unknown przez Groq ---")
    client = Groq(api_key=GROQ_API_KEY)

    prompt = f"""
Generate {n} short voice assistant commands (3–10 words each).

These must be OUT-OF-DISTRIBUTION (OOD) relative to SNIPS dataset.

SNIPS contains intents about:
- weather
- music playback
- restaurant booking
- taxi booking
- flight booking
- alarm setting
- general search / navigation

STRICT RULES:
- DO NOT generate anything related to: weather, music, alarms, reminders, restaurants, taxis, flights
- DO NOT paraphrase these intents
- DO NOT produce assistant-style commands like "set alarm", "play song", "book table"

Allowed OOD domains:
- legal or government info requests
- abstract questions (philosophy, definitions)
- technical instructions (coding, engineering concepts)
- medical information queries (non-appointment)
- financial explanations (not transactions or booking)
- academic/scientific questions

Mix:
- 50% near-OOD (still voice-command style but unrelated to SNIPS intents)
- 50% far-OOD (non-assistant-like informational queries)

Output format:
- one sentence per line
- no numbering
- no explanations
"""

    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    lines = completion.choices[0].message.content.strip().split('\n')

    cleaned = []
    for l in lines:
        l = l.strip()
        if 3 <= len(l.split()) <= 12:
            cleaned.append(l)

    return cleaned[:n]

In [ ]:
# ==========================================
# ŁADOWANIE I ŁĄCZENIE DANYCH (7 klas + 1)
# ==========================================
print("--- Pobieranie zbioru SNIPS (7 klas) ---")
raw_ds = load_dataset("DeepPavlov/snips", "default")
train_df = pd.DataFrame(raw_ds['train'])
test_df = pd.DataFrame(raw_ds['test'])
snips_df = pd.concat([train_df, test_df], ignore_index=True) # Zawiera klasy 0-6

possible_text_cols = ['query', 'utterance', 'text', 'sentence']
for col in possible_text_cols:
    if col in snips_df.columns:
        snips_df = snips_df.rename(columns={col: 'text'})
        break

# Pobieranie Unknown
unknown_texts = generate_unknown_data(NUM_OOD_SAMPLES)
OOD_CLASS_IDX = 7
unknown_df = pd.DataFrame({'text': unknown_texts, 'label': OOD_CLASS_IDX})

# Łączenie wszystkiego w jeden dataset
full_df = pd.concat([snips_df[['text', 'label']], unknown_df], ignore_index=True)
full_df.to_csv(f"{SAVE_PATH}/final_dataset_debug.csv", index=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

--- Pobieranie zbioru SNIPS (7 klas) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/366k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/43.3k [00:00<?, ?B/s]

...

--- Generowanie 1750 przykładów klasy Unknown przez Groq ---


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# ==========================================
# WALIDACJA LEAVE-TWO-OUT (5 FOLDS)
# ==========================================


FOLDS_CONFIG = [
    {"ood": [5, 6], "name": "Kino"},
    {"ood": [3, 0], "name": "Muzyka"},
    {"ood": [2, 1], "name": "Usługi"},
    {"ood": [4, 5], "name": "Twórczość"},
    {"ood": [6, 1], "name": "Mieszany"}
]


fold_results = []

for fold_idx, config in enumerate(FOLDS_CONFIG):
    ood_classes = config["ood"]
    id_classes = [c for c in range(7) if c not in ood_classes]

    print(f"\n>>> Rozpoczynanie Fałdu {fold_idx+1}/5: {config['name']}")
    print(f">>> Klasy ID (treningowe): {id_classes}")
    print(f">>> Klasy ukryte (OOD): {ood_classes}")


    # 1. Separujemy dane na ID (znane), OOD (ukryte) i Syntetyczne (Unknown)
    id_df_full = full_df[full_df['label'].isin(id_classes)].copy()
    ood_df_full = full_df[full_df['label'].isin(ood_classes)].copy()
    unknown_df_full = full_df[full_df['label'] == 7].copy()

    # 2. Dzielimy dane ID na próbki treningowe i testowe (np. 80/20)
    # Dzięki stratify zachowujemy proporcje podklas w treningu
    train_id, test_id = train_test_split(
        id_df_full, test_size=0.2, stratify=id_df_full['label'], random_state=42
    )


    # 4. SKŁADANIE FINALNYCH ZBIORÓW:
    # Trening: Widziane klasy ID + próbki Syntetyczne
    train_df_fold = pd.concat([train_id, unknown_df_full]).sample(frac=1, random_state=42)

    # Test: Niewidziane próbki ID + CAŁE ukryte OOD
    test_df_fold = pd.concat([test_id, ood_df_full]).sample(frac=1, random_state=42)

    mapping = {old_id: new_id for new_id, old_id in enumerate(id_classes)}
    mapping[7] = 5 # Klasa syntetyczna zawsze na ostatni indeks (5)

    train_fold_mapped = train_df_fold.copy()
    train_fold_mapped['mapped_label'] = train_fold_mapped['label'].map(mapping)

    train_ds = Dataset.from_pandas(train_fold_mapped[['text', 'mapped_label']].rename(columns={'mapped_label': 'label'})).map(tokenize_fn, batched=True)
    test_ds = Dataset.from_pandas(test_df_fold).map(tokenize_fn, batched=True)

    # Inicjalizacja modelu
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=6)

    training_args = TrainingArguments(
        output_dir=f"./temp_fold",
        num_train_epochs=3,
        per_device_train_batch_size=32,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=100,
        report_to="none",
        fp16=True
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=train_ds)
    trainer.train()

    # Przygotowanie do predykcji
    test_ds_for_predict = test_ds.remove_columns(["label"])

    # =========================
    # PREDYKCJA
    # =========================
    preds = trainer.predict(test_ds_for_predict)
    logits = preds.predictions
    true_labels = test_df_fold['label'].values

    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    pred_labels_mapped = np.argmax(probs, axis=1)
    ood_scores = probs[:, 5]

    # =========================
    # METRYKI OOD
    # =========================
    eval_mask = (true_labels != 7)
    y_true_binary = np.array([1 if l in ood_classes else 0 for l in true_labels[eval_mask]])
    y_scores = ood_scores[eval_mask]

    auroc = roc_auc_score(y_true_binary, y_scores)
    precision, recall, _ = precision_recall_curve(y_true_binary, y_scores)
    aupr = auc(recall, precision)

    fpr, tpr, thresholds = roc_curve(y_true_binary, y_scores)
    idx_95 = np.argmin(np.abs(tpr - 0.95))
    fpr95 = fpr[idx_95]

    # =========================
    # ZAPIS
    # =========================
    fold_df = pd.DataFrame({
        "text": test_df_fold["text"].values,
        "true_label_original": true_labels,
        "pred_label_mapped": pred_labels_mapped,
        "is_ood_true": [1 if l in ood_classes else 0 for l in true_labels],
        "ood_score": ood_scores,
        "fold": fold_idx + 1,
        "scenario": config['name']
    })

    fold_df.to_csv(f"{SAVE_PATH}/fold_{fold_idx+1}_predictions_exp3.csv", index=False)

    res = {
        "fold": fold_idx + 1,
        "scenario": config['name'],
        "auroc": auroc,
        "aupr": aupr,
        "fpr95": fpr95
    }
    fold_results.append(res)

    print(f"Fold {fold_idx+1} zakończony. AUROC (Real OOD): {auroc:.4f}")

    # Czyszczenie pamięci
    del model
    del trainer
    torch.cuda.empty_cache()


>>> Rozpoczynanie Fałdu 1/5: Kino
>>> Klasy ID (treningowe): [0, 1, 2, 3, 4]
>>> Klasy ukryte (OOD): [5, 6]


Map:   0%|          | 0/8482 [00:00<?, ? examples/s]

Map:   0%|          | 0/6188 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Map:   0%|          | 0/6203 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.391505
200,0.055783
300,0.034743
400,0.019295
500,0.014715
600,0.009375
700,0.005173


Fold 5 zakończony. AUROC (Real OOD): 0.8395


In [ ]:
# ==========================================
# PODSUMOWANIE I EKSPORT
# ==========================================
# Tworzymy główny DataFrame z wynikami wszystkich foldów
df_res = pd.DataFrame(fold_results)

# 1. ZAPIS WYNIKÓW SZCZEGÓŁOWYCH (Każdy fold/scenariusz osobno)
detailed_path = f"{SAVE_PATH}/wyniki_szczegolowe_per_fold_exp3.csv"
df_res.to_csv(detailed_path, index=False)
print(f"Zapisano wyniki szczegółowe dla każdego foldu w: {detailed_path}")

# 2. ZAPIS PODSUMOWANIA STATYSTYCZNEGO (Średnia + Odchylenie)
summary = {
    "Metric": ["AUROC", "AUPR", "FPR95"],
    "Mean": [df_res["auroc"].mean(), df_res["aupr"].mean(), df_res["fpr95"].mean()],
    "Std": [df_res["auroc"].std(), df_res["aupr"].std(), df_res["fpr95"].std()]
}
df_summary = pd.DataFrame(summary)
summary_path = f"{SAVE_PATH}/podsumowanie_statystyczne_exp3.csv"
df_summary.to_csv(summary_path, index=False)
print(f"Zapisano podsumowanie statystyczne w: {summary_path}")

# ==========================================
# WYŚWIETLENIE WYNIKÓW W KONSOLI
# ==========================================
print("\n" + "="*30)
print("RAPORT KOŃCOWY EKSPERYMENTU LEAVE 2 OUT")
print("="*30)
print("\n--- WYNIKI PER SCENARIUSZ (FOLD) ---")
print(df_res[["fold", "scenario", "auroc", "aupr", "fpr95"]].to_string(index=False))

print("\n--- ŚREDNIA I ODCHYLENIE (ŁĄCZNIE) ---")
print(df_summary.to_string(index=False))

Zapisano wyniki szczegółowe dla każdego foldu w: /content/drive/MyDrive/SNIPS_OOD_Project/wyniki_szczegolowe_per_fold_exp3.csv
Zapisano podsumowanie statystyczne w: /content/drive/MyDrive/SNIPS_OOD_Project/podsumowanie_statystyczne_exp3.csv

RAPORT KOŃCOWY EKSPERYMENTU LEAVE 2 OUT

--- WYNIKI PER SCENARIUSZ (FOLD) ---
 fold  scenario    auroc     aupr    fpr95
    1      Kino 0.909866 0.962747 0.775422
    2    Muzyka 0.612373 0.783893 0.980184
    3    Usługi 0.760015 0.878935 0.870577
    4 Twórczość 0.926382 0.969495 0.631807
    5  Mieszany 0.839490 0.929410 0.774988

--- ŚREDNIA I ODCHYLENIE (ŁĄCZNIE) ---
Metric     Mean      Std
 AUROC 0.809625 0.128396
  AUPR 0.904896 0.076545
 FPR95 0.806595 0.129178


In [ ]:
from sklearn.model_selection import train_test_split
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

# ==========================================
# WALIDACJA LEAVE-ONE-OUT (7 FOLDS)
# ==========================================

NUM_REAL_CLASSES = 7  # Klasy od 0 do 6
fold_results = []

for ood_class_idx in range(NUM_REAL_CLASSES):
    # Definicja klas dla tego foldu
    ood_classes = [ood_class_idx]
    id_classes = [c for c in range(NUM_REAL_CLASSES) if c != ood_class_idx]

    # Nazwa scenariusza na podstawie wyrzuconej klasy
    scenario_name = f"OOD_Class_{ood_class_idx}"

    print(f"\n>>> Rozpoczynanie Fałdu {ood_class_idx + 1}/{NUM_REAL_CLASSES}: {scenario_name}")
    print(f">>> Klasy ID (treningowe): {id_classes}")
    print(f">>> Klasa ukryta (OOD): {ood_classes}")

    # 1. Separujemy dane
    id_df_full = full_df[full_df['label'].isin(id_classes)].copy()
    ood_df_full = full_df[full_df['label'].isin(ood_classes)].copy()
    unknown_df_full = full_df[full_df['label'] == 7].copy()

    # 2. Split ID na train/test
    train_id, test_id = train_test_split(
        id_df_full, test_size=0.2, stratify=id_df_full['label'], random_state=42
    )

    # 3. Składanie zbiorów
    # Trening: 6 klas ID + klasa syntetyczna (7)
    train_df_fold = pd.concat([train_id, unknown_df_full]).sample(frac=1, random_state=42)
    # Test: Reszta z 6 klas ID + CAŁA wyrzucona klasa OOD
    test_df_fold = pd.concat([test_id, ood_df_full]).sample(frac=1, random_state=42)

    # 4. Mapowanie etykiet (0-5 dla ID, 6 dla syntetycznej)
    mapping = {old_id: new_id for new_id, old_id in enumerate(id_classes)}
    mapping[7] = len(id_classes)  # Klasa syntetyczna ląduje na indeksie 6

    train_fold_mapped = train_df_fold.copy()
    train_fold_mapped['mapped_label'] = train_fold_mapped['label'].map(mapping)

    # Przygotowanie Datasetów
    train_ds = Dataset.from_pandas(
        train_fold_mapped[['text', 'mapped_label']].rename(columns={'mapped_label': 'label'})
    ).map(tokenize_fn, batched=True)

    test_ds = Dataset.from_pandas(test_df_fold).map(tokenize_fn, batched=True)

    # 5. Inicjalizacja modelu (num_labels = 6 ID + 1 Syntetyczna = 7)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(id_classes) + 1)

    training_args = TrainingArguments(
        output_dir=f"./temp_fold_loo",
        num_train_epochs=3,
        per_device_train_batch_size=32,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=100,
        report_to="none",
        fp16=True
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=train_ds)
    trainer.train()

    # 6. Predykcja
    test_ds_for_predict = test_ds.remove_columns(["label"])
    preds = trainer.predict(test_ds_for_predict)
    logits = preds.predictions
    true_labels = test_df_fold['label'].values

    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    pred_labels_mapped = np.argmax(probs, axis=1)

    # Wynik OOD to prawdopodobieństwo klasy "Unknown" (ostatni indeks)
    ood_scores = probs[:, len(id_classes)]

    # 7. Metryki (wykluczamy syntetyczne z testu, jeśli by tam były -
    # w tym kodzie test_id ich nie ma, ale zachowujemy maskę dla bezpieczeństwa)
    eval_mask = (true_labels != 7)
    y_true_binary = np.array([1 if l in ood_classes else 0 for l in true_labels[eval_mask]])
    y_scores = ood_scores[eval_mask]

    auroc = roc_auc_score(y_true_binary, y_scores)
    precision, recall, _ = precision_recall_curve(y_true_binary, y_scores)
    aupr = auc(recall, precision)

    fpr, tpr, thresholds = roc_curve(y_true_binary, y_scores)
    idx_95 = np.argmin(np.abs(tpr - 0.95))
    fpr95 = fpr[idx_95]

    # 8. Zapis wyników cząstkowych
    fold_df = pd.DataFrame({
        "text": test_df_fold["text"].values,
        "true_label_original": true_labels,
        "pred_label_mapped": pred_labels_mapped,
        "is_ood_true": [1 if l in ood_classes else 0 for l in true_labels],
        "ood_score": ood_scores,
        "fold": ood_class_idx + 1,
        "scenario": scenario_name
    })
    fold_df.to_csv(f"{SAVE_PATH}/fold_loo_{ood_class_idx+1}_predictions.csv", index=False)

    res = {
        "fold": ood_class_idx + 1,
        "scenario": scenario_name,
        "auroc": auroc,
        "aupr": aupr,
        "fpr95": fpr95
    }
    fold_results.append(res)

    print(f"Fold {ood_class_idx+1} zakończony. Klasa OOD: {ood_class_idx} | AUROC: {auroc:.4f}")

    # Czyszczenie
    del model
    del trainer
    torch.cuda.empty_cache()




>>> Rozpoczynanie Fałdu 1/7: OOD_Class_0
>>> Klasy ID (treningowe): [1, 2, 3, 4, 5, 6]
>>> Klasa ukryta (OOD): [0]


Map:   0%|          | 0/10139 [00:00<?, ? examples/s]

Map:   0%|          | 0/4531 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Map:   0%|          | 0/4544 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.502361
200,0.056198
300,0.047409
400,0.023662
500,0.011457
600,0.018637
700,0.015592
800,0.007372
900,0.004184


Fold 7 zakończony. Klasa OOD: 6 | AUROC: 0.7613


In [ ]:
# ==========================================
# PODSUMOWANIE I EKSPORT (LOO)
# ==========================================
df_res = pd.DataFrame(fold_results)

# Zapis wyników
df_res.to_csv(f"{SAVE_PATH}/wyniki_loo_per_class.csv", index=False)

summary = {
    "Metric": ["AUROC", "AUPR", "FPR95"],
    "Mean": [df_res["auroc"].mean(), df_res["aupr"].mean(), df_res["fpr95"].mean()],
    "Std": [df_res["auroc"].std(), df_res["aupr"].std(), df_res["fpr95"].std()]
}
df_summary = pd.DataFrame(summary)
df_summary.to_csv(f"{SAVE_PATH}/podsumowanie_statystyczne_loo.csv", index=False)

print("\n--- RAPORT KOŃCOWY LEAVE-ONE-OUT ---")
print(df_res[["fold", "scenario", "auroc", "fpr95"]].to_string(index=False))
print("\n--- STATYSTYKI ŁĄCZNE ---")
print(df_summary.to_string(index=False))


--- RAPORT KOŃCOWY LEAVE-ONE-OUT ---
 fold    scenario    auroc    fpr95
    1 OOD_Class_0 0.695715 0.541985
    2 OOD_Class_1 0.954584 0.372130
    3 OOD_Class_2 0.966495 0.072265
    4 OOD_Class_3 0.895972 0.501009
    5 OOD_Class_4 0.906451 0.655672
    6 OOD_Class_5 0.849806 0.742961
    7 OOD_Class_6 0.761258 0.979074

--- STATYSTYKI ŁĄCZNE ---
Metric     Mean      Std
 AUROC 0.861469 0.100472
  AUPR 0.861704 0.118318
 FPR95 0.552157 0.287172


In [ ]:
# ==========================================
# EKSPERYMENT "ONE-VS-REST" (ODWRÓCONY)
# ==========================================
# Uczymy na 1 wybranej klasie realnej + klasie Syntetycznej (Unknown).
# Testujemy na reszcie świata (pozostałe 6 klas jako OOD).

fold_results_ovr = []

for target_class_idx in range(NUM_REAL_CLASSES):
    # Definicja ról dla tego foldu
    id_class = [target_class_idx]
    ood_classes = [c for c in range(NUM_REAL_CLASSES) if c != target_class_idx]

    # Nazwa scenariusza dla jasności w logach i plikach
    scenario_name = f"Train_on_class_{target_class_idx}_vs_Rest"

    print(f"\n>>> Rozpoczynanie Eksperymentu Odwróconego {target_class_idx + 1}/7")
    print(f">>> Klasa ID (treningowa): {id_class}")
    print(f">>> Klasy testowane jako OOD: {ood_classes}")

    # 1. Separujemy dane
    id_df_full = full_df[full_df['label'] == target_class_idx].copy()
    ood_df_full = full_df[full_df['label'].isin(ood_classes)].copy()
    unknown_df_full = full_df[full_df['label'] == 7].copy()

    # 2. Split ID na train/test (80/20)
    train_id, test_id = train_test_split(
        id_df_full, test_size=0.2, stratify=id_df_full['label'], random_state=42
    )

    # 3. Składanie finalnych zbiorów dla tego foldu
    # Trening: 1 klasa ID + Syntetyczne (label 7)
    train_df_fold = pd.concat([train_id, unknown_df_full]).sample(frac=1, random_state=42)
    # Test: Próbki testowe ID + WSZYSTKIE pozostałe klasy OOD
    test_df_fold = pd.concat([test_id, ood_df_full]).sample(frac=1, random_state=42)

    # 4. Mapowanie etykiet (0: Znana klasa, 1: Unknown/Syntetyczna)
    mapping = {target_class_idx: 0, 7: 1}

    train_fold_mapped = train_df_fold.copy()
    train_fold_mapped['mapped_label'] = train_fold_mapped['label'].map(mapping)

    # Przygotowanie Datasetów HuggingFace
    train_ds = Dataset.from_pandas(
        train_fold_mapped[['text', 'mapped_label']].rename(columns={'mapped_label': 'label'})
    ).map(tokenize_fn, batched=True)

    test_ds = Dataset.from_pandas(test_df_fold).map(tokenize_fn, batched=True)

    # 5. Inicjalizacja modelu (Problem binarny: Znana vs Syntetyczna)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    training_args = TrainingArguments(
        output_dir=f"./temp_fold_ovr",
        num_train_epochs=3,
        per_device_train_batch_size=32,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=100,
        report_to="none",
        fp16=True
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=train_ds)
    trainer.train()

    # 6. Predykcja na zbiorze testowym
    test_ds_for_predict = test_ds.remove_columns(["label"])
    preds = trainer.predict(test_ds_for_predict)
    logits = preds.predictions
    true_labels = test_df_fold['label'].values

    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    pred_labels_mapped = np.argmax(probs, axis=1)

    # Wynik OOD to pewność modelu co do klasy "1" (Unknown)
    ood_scores = probs[:, 1]

    # 7. Obliczanie metryk OOD
    # Chcemy sprawdzić, jak ood_score (prawd. klasy syntetycznej) odróżnia ID od realnego OOD
    y_true_binary = np.array([1 if l in ood_classes else 0 for l in true_labels])
    y_scores = ood_scores

    auroc = roc_auc_score(y_true_binary, y_scores)
    precision, recall, _ = precision_recall_curve(y_true_binary, y_scores)
    aupr = auc(recall, precision)

    fpr, tpr, thresholds = roc_curve(y_true_binary, y_scores)
    idx_95 = np.argmin(np.abs(tpr - 0.95))
    fpr95 = fpr[idx_95]

    # 8. ZAPIS PEŁNEJ INFORMACJI (bez wycinania kolumn)
    fold_df = pd.DataFrame({
        "text": test_df_fold["text"].values,
        "true_label_original": true_labels,
        "pred_label_mapped": pred_labels_mapped,
        "is_ood_true": y_true_binary,
        "ood_score": ood_scores,
        "fold": target_class_idx + 1,
        "scenario": scenario_name,
        "trained_on_id_class": target_class_idx
    })

    # Zapis każdego foldu do osobnego pliku
    fold_df.to_csv(f"{SAVE_PATH}/fold_ovr_{target_class_idx}_predictions_full.csv", index=False)

    # Kolekcjonowanie wyników zbiorczych
    res = {
        "fold": target_class_idx + 1,
        "scenario": scenario_name,
        "trained_on": target_class_idx,
        "auroc": auroc,
        "aupr": aupr,
        "fpr95": fpr95
    }
    fold_results_ovr.append(res)

    print(f"Zakończono: {scenario_name} | AUROC: {auroc:.4f}")

    # Czyszczenie zasobów GPU
    del model
    del trainer
    torch.cuda.empty_cache()



>>> Rozpoczynanie Eksperymentu Odwróconego 1/7
>>> Klasa ID (treningowa): [0]
>>> Klasy testowane jako OOD: [1, 2, 3, 4, 5, 6]


Map:   0%|          | 0/1819 [00:00<?, ? examples/s]

Map:   0%|          | 0/12851 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Map:   0%|          | 0/12837 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.024738


Zakończono: Train_on_class_6_vs_Rest | AUROC: 0.9407


In [ ]:
# ==========================================
# PODSUMOWANIE I EKSPORT WYNIKÓW OVR
# ==========================================
df_res_ovr = pd.DataFrame(fold_results_ovr)

# 1. Zapis wyników szczegółowych dla wszystkich foldów OVR
detailed_path_ovr = f"{SAVE_PATH}/wyniki_szczegolowe_ovr_exp.csv"
df_res_ovr.to_csv(detailed_path_ovr, index=False)

# 2. Zapis podsumowania statystycznego
summary_ovr = {
    "Metric": ["AUROC", "AUPR", "FPR95"],
    "Mean": [df_res_ovr["auroc"].mean(), df_res_ovr["aupr"].mean(), df_res_ovr["fpr95"].mean()],
    "Std": [df_res_ovr["auroc"].std(), df_res_ovr["aupr"].std(), df_res_ovr["fpr95"].std()]
}
df_summary_ovr = pd.DataFrame(summary_ovr)
df_summary_ovr.to_csv(f"{SAVE_PATH}/podsumowanie_statystyczne_ovr.csv", index=False)

print("\n" + "="*30)
print("RAPORT KOŃCOWY ONE-VS-REST")
print("="*30)
print(df_res_ovr[["fold", "trained_on", "auroc", "fpr95"]].to_string(index=False))
print("\n--- ŚREDNIE WYNIKI OVR ---")
print(df_summary_ovr.to_string(index=False))


RAPORT KOŃCOWY ONE-VS-REST
 fold  trained_on    auroc    fpr95
    1           0 0.989575 0.034230
    2           1 0.995896 0.019277
    3           2 0.994195 0.028571
    4           3 0.977506 0.100000
    5           4 0.999239 0.002427
    6           5 0.744203 0.693431
    7           6 0.940739 0.177184

--- ŚREDNIE WYNIKI OVR ---
Metric     Mean      Std
 AUROC 0.948765 0.092407
  AUPR 0.997511 0.004766
 FPR95 0.150732 0.246835
